# LDA to IRS matching: name and state

This notebook reads Parquet directly from S3 and evaluates IRS organization-to-LDA client candidates using normalized organization names and available state evidence. It is read-only: candidates remain reviewable links rather than approved identity assertions.

IRS state comes from the EIN-level `irs_master` table. The LDA **client** is the organization being represented; the **registrant** is often an outside lobbying firm, so registrant addresses are not used.

In [133]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_BASE = "s3://irs-990-263839540825-us-east-2-an/parquet"
AWS_REGION = "us-east-2"

conn = duckdb.connect(database=":memory:")
conn.execute("INSTALL httpfs; LOAD httpfs;")
conn.execute(f"""
    CREATE SECRET s3_access (
        TYPE s3,
        PROVIDER credential_chain,
        REGION '{AWS_REGION}'
    )
""")

for table in ("organizations", "irs_master", "lda_filings", "entity_observations"):
    conn.execute(f"""
        CREATE VIEW {table} AS
        SELECT *
        FROM read_parquet(
            '{PARQUET_BASE}/{table}/*.parquet',
            union_by_name = true
        )
    """)

def q(sql):
    return conn.execute(sql).df()

coverage = q("""
SELECT 1 AS sort_order, 'Form 990 e-file organization EINs' AS metric,
       COUNT(DISTINCT ein) AS value
FROM organizations
UNION ALL
SELECT 2, 'EINs in current loaded IRS master extract', COUNT(DISTINCT ein)
FROM irs_master
UNION ALL
SELECT 3, 'Current IRS master EINs with state', COUNT(DISTINCT ein)
FROM irs_master
WHERE NULLIF(TRIM(state), '') IS NOT NULL
UNION ALL
SELECT 4, 'E-file organization EINs found in current IRS master',
       COUNT(DISTINCT organization.ein)
FROM organizations AS organization
JOIN irs_master AS master USING (ein)
UNION ALL
SELECT 5, 'E-file organization EINs not found in current IRS master',
       COUNT(DISTINCT organization.ein)
FROM organizations AS organization
LEFT JOIN irs_master AS master USING (ein)
WHERE master.ein IS NULL
UNION ALL
SELECT 6, 'LDA filings', COUNT(*) FROM lda_filings
UNION ALL
SELECT 7, 'LDA filings with client state', COUNT(*)
FROM lda_filings
WHERE json_valid(raw_json)
  AND NULLIF(TRIM(json_extract_string(raw_json, '$.client.state')), '') IS NOT NULL
UNION ALL
SELECT 8, 'Distinct LDA client IDs',
       COUNT(DISTINCT json_extract_string(raw_json, '$.client.id'))
FROM lda_filings WHERE json_valid(raw_json)
UNION ALL
SELECT 9, 'Distinct LDA client IDs with state',
       COUNT(DISTINCT CASE
           WHEN NULLIF(TRIM(json_extract_string(raw_json, '$.client.state')), '') IS NOT NULL
           THEN json_extract_string(raw_json, '$.client.id') END)
FROM lda_filings WHERE json_valid(raw_json)
ORDER BY sort_order
""").drop(columns="sort_order")

print(f"DuckDB reading Parquet from: {PARQUET_BASE}")
coverage.style.format({"value": "{:,.0f}"})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB reading Parquet from: s3://irs-990-263839540825-us-east-2-an/parquet


,metric,value
0,Form 990 e-file organization EINs,"861,081"
1,EINs in current loaded IRS master extract,"279,802"
2,Current IRS master EINs with state,"279,802"
3,E-file organization EINs found in current IRS master,"130,626"
4,E-file organization EINs not found in current IRS master,"730,455"
5,LDA filings,"192,174"
6,LDA filings with client state,"186,203"
7,Distinct LDA client IDs,"28,790"
8,Distinct LDA client IDs with state,"27,845"


## Match definitions

The analysis uses compact EIN-level IRS records and deduplicated LDA clients:

- **Identical normalized-name link:** at least one IRS EIN has exactly the same normalized-name string as the LDA client. This is an organization-level candidate link, even if state is unavailable or multiple EINs remain.
- **Name + state link:** an identical normalized-name link where `irs_master.state` equals the LDA client state.
- **State-confirmed link:** at least one IRS candidate has an identical normalized name and an agreeing state. Multiple agreeing EINs are allowed at this organization-link stage.
- **No identical normalized name:** the equality join found no identical normalized string. This is the input to the subsequent near-name candidate stage, not evidence that the organization is absent from IRS data.
- Missing state is unknown and is never interpreted as disagreement.
- A state conflict is a review signal, not proof of a false match; organizations can move, LDA may report a government-relations office, and IRS master data can lag.

In [134]:
conn.execute("""
CREATE OR REPLACE TEMP TABLE irs_entities AS
SELECT
    organization.ein,
    organization.current_name AS irs_name,
    organization.normalized_name,
    master.ein IS NOT NULL AS irs_master_present,
    UPPER(NULLIF(TRIM(master.state), '')) AS irs_state
FROM organizations AS organization
LEFT JOIN irs_master AS master USING (ein)
WHERE organization.normalized_name IS NOT NULL
  AND organization.normalized_name <> ''
""")

conn.execute("""
CREATE OR REPLACE TEMP TABLE lda_clients AS
SELECT DISTINCT
    json_extract_string(filing.raw_json, '$.client.id') AS client_id,
    filing.client_name AS lda_name,
    observation.normalized_name,
    UPPER(NULLIF(TRIM(
        json_extract_string(filing.raw_json, '$.client.state')
    ), '')) AS lda_state
FROM entity_observations AS observation
JOIN lda_filings AS filing
  ON filing.filing_uuid = observation.source_record_id
WHERE observation.source_system = 'LDA'
  AND observation.normalized_name <> ''
  AND json_valid(filing.raw_json)
  AND json_extract_string(filing.raw_json, '$.client.id') IS NOT NULL
""")

conn.execute("""
CREATE OR REPLACE TEMP TABLE name_state_links AS
SELECT DISTINCT
    irs.ein,
    lda.client_id,
    irs.irs_name,
    lda.lda_name,
    irs.normalized_name,
    irs.irs_master_present,
    irs.irs_state,
    lda.lda_state
FROM irs_entities AS irs
JOIN lda_clients AS lda USING (normalized_name)
""")

match_summary = q("""
SELECT 'Distinct name-only EIN-client links' AS metric, COUNT(*) AS value
FROM name_state_links
UNION ALL
SELECT 'Name-only links with both states available', COUNT(*)
FROM name_state_links
WHERE irs_state IS NOT NULL AND lda_state IS NOT NULL
UNION ALL
SELECT 'Name + state agreeing links', COUNT(*)
FROM name_state_links
WHERE irs_state = lda_state AND irs_state IS NOT NULL
UNION ALL
SELECT 'Name matches with conflicting states', COUNT(*)
FROM name_state_links
WHERE irs_state <> lda_state
  AND irs_state IS NOT NULL AND lda_state IS NOT NULL
UNION ALL
SELECT 'Distinct EINs in name + state links', COUNT(DISTINCT ein)
FROM name_state_links
WHERE irs_state = lda_state AND irs_state IS NOT NULL
UNION ALL
SELECT 'Distinct LDA clients in name + state links', COUNT(DISTINCT client_id)
FROM name_state_links
WHERE irs_state = lda_state AND irs_state IS NOT NULL
""")
match_summary.style.format({"value": "{:,.0f}"})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,metric,value
0,Distinct name-only EIN-client links,"12,922"
1,Name-only links with both states available,"1,704"
2,Name + state agreeing links,590
3,Name matches with conflicting states,"1,114"
4,Distinct EINs in name + state links,440
5,Distinct LDA clients in name + state links,564


## State availability

These tables represent different IRS products and time scopes. `organizations` is built from EINs observed in parsed Form 990 e-file XML, while `irs_master` is loaded separately from the BMF CSV file or files supplied to the loader. In the current Parquet snapshot:

- **130,626 EINs** occur in both tables.
- **730,455 EINs** occur in the Form 990 e-file organization table but not in the current loaded master extract.
- **149,176 EINs** occur in the current master extract but have no parsed e-file organization row.

Therefore, "not found in current IRS master" means only that the left join found no row in this particular loaded extract. It does not mean the EIN is invalid, lacks tax-exempt status, or has no state in every IRS source. All 279,802 rows in the current master extract have state, but that state can enrich only overlapping EINs.

The LDA client state is nearly complete. Do not fill a missing LDA client state with the registrant state because the registrant is frequently an outside lobbying firm in Washington, DC.

In [ ]:
state_availability = q("""
SELECT
    CASE
        WHEN NOT irs_master_present AND lda_state IS NULL
            THEN 'IRS master row absent; LDA state missing'
        WHEN NOT irs_master_present
            THEN 'IRS master row absent'
        WHEN irs_state IS NULL AND lda_state IS NULL
            THEN 'IRS master state blank; LDA state missing'
        WHEN irs_state IS NULL
            THEN 'IRS master state blank'
        WHEN lda_state IS NULL
            THEN 'LDA state missing'
        ELSE 'Both states available'
    END AS availability,
    COUNT(*) AS links,
    COUNT(DISTINCT ein) AS eins,
    COUNT(DISTINCT client_id) AS lda_clients,
    COUNT(DISTINCT normalized_name) AS normalized_names
FROM name_state_links
GROUP BY availability
ORDER BY links DESC
""")
state_availability.style.format({
    "links": "{:,.0f}",
    "eins": "{:,.0f}",
    "lda_clients": "{:,.0f}",
    "normalized_names": "{:,.0f}",
})

,availability,links,eins,lda_clients,normalized_names
0,IRS master row absent,"11,166","7,002","5,051","3,209"
1,Both states available,"1,704","1,157",860,587
2,IRS master row absent; LDA state missing,47,46,41,40
3,LDA state missing,5,5,5,5


## Confidence tiers

A unique normalized name is stronger than a chaptered or federated name. State can confirm a unique-name link or narrow a multi-EIN name, but links that remain multi-EIN after state agreement still require review.

In [ ]:
conn.execute("""
CREATE OR REPLACE TEMP TABLE classified_links AS
WITH name_state_counts AS (
    SELECT
        normalized_name,
        lda_state,
        COUNT(DISTINCT ein) AS all_eins,
        COUNT(DISTINCT ein) FILTER (
            WHERE irs_state = lda_state AND irs_state IS NOT NULL
        ) AS agreeing_eins
    FROM name_state_links
    GROUP BY normalized_name, lda_state
)
SELECT
    link.*,
    count.all_eins,
    count.agreeing_eins,
    CASE
        WHEN count.all_eins = 1
         AND link.irs_state = link.lda_state
         AND link.irs_state IS NOT NULL
            THEN 'A: unique name + state agrees'
        WHEN count.all_eins > 1
         AND count.agreeing_eins = 1
         AND link.irs_state = link.lda_state
         AND link.irs_state IS NOT NULL
            THEN 'B: state resolves multi-EIN name'
        WHEN link.irs_state = link.lda_state
         AND link.irs_state IS NOT NULL
            THEN 'C: exact name + state match; multiple IRS EINs'
        WHEN link.irs_state IS NULL OR link.lda_state IS NULL
            THEN 'D: name only; state unavailable'
        ELSE 'E: name matches but state conflicts'
    END AS confidence_tier
FROM name_state_links AS link
JOIN name_state_counts AS count
  ON count.normalized_name = link.normalized_name
 AND count.lda_state IS NOT DISTINCT FROM link.lda_state
""")

confidence_tiers = q("""
SELECT
    confidence_tier,
    COUNT(*) AS links,
    COUNT(DISTINCT normalized_name) AS normalized_names,
    COUNT(DISTINCT ein) AS eins,
    COUNT(DISTINCT client_id) AS lda_clients
FROM classified_links
GROUP BY confidence_tier
ORDER BY confidence_tier
""")

confidence_tiers.style.format({
    "links": "{:,.0f}",
    "normalized_names": "{:,.0f}",
    "eins": "{:,.0f}",
    "lda_clients": "{:,.0f}",
})

,confidence_tier,links,normalized_names,eins,lda_clients
0,A: unique name + state agrees,517,380,380,517
1,B: state resolves multi-EIN name,25,22,22,25
2,C: exact name + state match; multiple IRS EINs,48,17,38,22
3,D: name only; state unavailable,"11,218","3,235","7,029","5,097"
4,E: name matches but state conflicts,"1,114",203,748,301


In [ ]:
review_examples = q("""
WITH ranked AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY confidence_tier
        ORDER BY normalized_name, ein, client_id
    ) AS example_number
    FROM classified_links
)
SELECT
    confidence_tier,
    ein,
    client_id,
    irs_name,
    lda_name,
    normalized_name,
    irs_state,
    lda_state,
    all_eins,
    agreeing_eins
FROM ranked
WHERE example_number <= 5
ORDER BY confidence_tier, example_number
""")
review_examples

,confidence_tier,ein,client_id,irs_name,lda_name,normalized_name,irs_state,lda_state,all_eins,agreeing_eins
0,A: unique name + state agrees,222130220,156991,180 TURNING LIVES AROUND,"180 TURNING LIVES AROUND, INC",180 turning lives around,NJ,NJ,1,1
1,A: unique name + state agrees,710866051,174982,THE AFIKIM FOUNDATION,AFIKIM FOUNDATION,afikim foundation,NY,NY,1,1
2,A: unique name + state agrees,521622589,101344,ALBANIAN AMERICAN CIVIC LEAGUE INC,ALBANIAN AMERICAN CIVIC LEAGUE,albanian american civic league,NY,NY,1,1
3,A: unique name + state agrees,160743900,209381,ALFRED UNIVERSITY,ALFRED UNIVERSITY,alfred university,NY,NY,1,1
4,A: unique name + state agrees,020222791,175930,Alice Peck Day Memorial Hospital,ALICE PECK DAY MEMORIAL HOSPITAL,alice peck day memorial hospital,NH,NH,1,1
5,B: state resolves multi-EIN name,474608840,58437,AMERICAN PROMISE INC,"AMERICAN PROMISE, INC.",american promise,MA,MA,2,1
6,B: state resolves multi-EIN name,421182936,52975,American Society Of Transplantation,AMERICAN SOCIETY OF TRANSPLANTATION,american society transplantation,NJ,NJ,2,1
7,B: state resolves multi-EIN name,131923626,200694,THE CARNEGIE HALL CORPORATION,CARNEGIE HALL CORPORATION,carnegie hall,NY,NY,2,1
8,B: state resolves multi-EIN name,222746879,203086,CAST INC,CAST,cast,MA,MA,2,1
9,B: state resolves multi-EIN name,300553416,69580,CATHOLIC CHARITIES OF THE,CATHOLIC CHARITIES,catholic charities,NY,NY,19,1


## Downstream policy sample

Tier A is the narrow primary cohort. Tier A+B is a broader sensitivity cohort where state resolves a name that otherwise maps to multiple EINs. The following cell measures how much LDA filing and bill-reference coverage those cohorts provide.

In [ ]:
conn.execute(f"""
CREATE VIEW lobbying_bill_links AS
SELECT *
FROM read_parquet(
    '{PARQUET_BASE}/lobbying_bill_links/*.parquet',
    union_by_name = true
)
""")

policy_sample = q("""
WITH cohorts AS (
    SELECT 'Tier A only' AS cohort, *
    FROM classified_links
    WHERE confidence_tier = 'A: unique name + state agrees'
    UNION ALL
    SELECT 'Tier A + B' AS cohort, *
    FROM classified_links
    WHERE confidence_tier IN (
        'A: unique name + state agrees',
        'B: state resolves multi-EIN name'
    )
),
client_filings AS (
    SELECT DISTINCT
        cohort.cohort,
        cohort.ein,
        cohort.client_id,
        filing.filing_uuid,
        filing.filing_year
    FROM cohorts AS cohort
    JOIN lda_filings AS filing
      ON json_extract_string(filing.raw_json, '$.client.id') = cohort.client_id
),
bill_exposure AS (
    SELECT DISTINCT
        client.cohort,
        client.ein,
        client.client_id,
        client.filing_uuid,
        client.filing_year,
        bill.bill_type,
        bill.bill_number
    FROM client_filings AS client
    JOIN lobbying_bill_links AS bill USING (filing_uuid)
)
SELECT
    cohort,
    COUNT(DISTINCT ein) AS eins,
    COUNT(DISTINCT client_id) AS lda_clients,
    COUNT(DISTINCT filing_uuid) AS lda_filings,
    COUNT(*) AS filing_bill_references,
    COUNT(DISTINCT (
        CAST((filing_year - 1789) // 2 AS INTEGER) + 1,
        bill_type,
        bill_number
    )) AS distinct_bills
FROM bill_exposure
GROUP BY cohort
ORDER BY cohort
""")
policy_sample.style.format({
    "eins": "{:,.0f}",
    "lda_clients": "{:,.0f}",
    "lda_filings": "{:,.0f}",
    "filing_bill_references": "{:,.0f}",
    "distinct_bills": "{:,.0f}",
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,cohort,eins,lda_clients,lda_filings,filing_bill_references,distinct_bills
0,Tier A + B,150,185,"1,038","5,417",956
1,Tier A only,140,174,959,"5,071",917


In [ ]:
conn.execute("""
CREATE OR REPLACE TEMP TABLE lda_client_match_status AS
WITH candidate_counts AS (
    SELECT
        client.client_id,
        COUNT(DISTINCT candidate.ein) AS identical_name_eins,
        COUNT(DISTINCT candidate.ein) FILTER (
            WHERE candidate.irs_state = client.lda_state
              AND candidate.irs_state IS NOT NULL
        ) AS state_agreeing_eins,
        COUNT(DISTINCT candidate.ein) FILTER (
            WHERE candidate.irs_state IS NOT NULL
              AND client.lda_state IS NOT NULL
        ) AS comparable_state_eins
    FROM lda_clients AS client
    LEFT JOIN irs_entities AS candidate
      ON candidate.normalized_name = client.normalized_name
    GROUP BY client.client_id
)
SELECT
    client_id,
    identical_name_eins,
    state_agreeing_eins,
    comparable_state_eins,
    CASE
        WHEN state_agreeing_eins > 0
            THEN 'Identical normalized name: state agrees'
        WHEN identical_name_eins = 0
            THEN 'No identical normalized name'
        WHEN comparable_state_eins = 0
            THEN 'Identical normalized name: state unavailable'
        ELSE 'Identical normalized name: state conflicts'
    END AS match_status
FROM candidate_counts
""")

exact_name_misses = q("""
WITH lda_client_filings AS (
    SELECT
        json_extract_string(filing.raw_json, '$.client.id') AS client_id,
        filing.client_name AS lda_name,
        observation.normalized_name AS lda_normalized_name,
        UPPER(NULLIF(TRIM(
            json_extract_string(filing.raw_json, '$.client.state')
        ), '')) AS lda_state,
        filing.filing_uuid,
        filing.filing_year,
        filing.registrant_name
    FROM entity_observations AS observation
    JOIN lda_filings AS filing
      ON filing.filing_uuid = observation.source_record_id
    WHERE observation.source_system = 'LDA'
      AND observation.subject_role = 'client'
      AND NULLIF(observation.normalized_name, '') IS NOT NULL
      AND json_valid(filing.raw_json)
      AND json_extract_string(filing.raw_json, '$.client.id') IS NOT NULL
)
SELECT
    client.client_id,
    ARG_MAX(client.lda_name, client.filing_year) AS lda_name,
    ARG_MAX(client.lda_normalized_name, client.filing_year) AS lda_normalized_name,
    ARG_MAX(client.lda_state, client.filing_year) AS lda_state,
    COUNT(DISTINCT client.filing_uuid) AS lda_filings,
    MIN(client.filing_year) AS first_filing_year,
    MAX(client.filing_year) AS latest_filing_year,
    COUNT(DISTINCT client.registrant_name) AS registrants
FROM lda_client_filings AS client
JOIN lda_client_match_status AS status USING (client_id)
WHERE status.identical_name_eins = 0
GROUP BY client.client_id
ORDER BY lda_filings DESC, lda_name, client.client_id
""")

client_match_summary = q("""
SELECT
    match_status,
    COUNT(*) AS lda_clients
FROM lda_client_match_status
GROUP BY match_status
ORDER BY
    CASE match_status
        WHEN 'Identical normalized name: state agrees' THEN 1
        WHEN 'Identical normalized name: state unavailable' THEN 2
        WHEN 'Identical normalized name: state conflicts' THEN 3
        ELSE 4
    END
""")

print(
    f"Clients requiring near-name search: {len(exact_name_misses):,} "
    f"of {int(client_match_summary['lda_clients'].sum()):,}"
)
display(client_match_summary.style.format({"lda_clients": "{:,.0f}"}))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clients requiring near-name search: 23,030 of 28,790


,match_status,lda_clients
0,Identical normalized name: state agrees,564
1,Identical normalized name: state unavailable,"4,900"
2,Identical normalized name: state conflicts,296
3,No identical normalized name,"23,030"


## Near-name candidate generation

The exact-string baseline above is intentionally conservative. This stage searches clients without an identical normalized-name candidate using a blocked Jaro-Winkler comparison:

- Names must share their first eight normalized characters.
- The shorter name must be at least 60% of the longer name.
- Jaro-Winkler similarity must be at least 0.90.
- State agreement adjusts ranking but does not override weak name evidence.
- Results are **candidates for review**, not approved EIN links.

Blocking avoids a full cross join between every LDA client and every IRS EIN. The best-candidate table retains one top-ranked EIN per client while the full candidate table remains available for ambiguity review.

In [ ]:
NEAR_NAME_MIN_SIMILARITY = 0.90
NEAR_NAME_STRONG_SIMILARITY = 0.94
NEAR_NAME_CONFLICT_SIMILARITY = 0.97

conn.execute(f"""
CREATE OR REPLACE TEMP TABLE near_name_candidates AS
WITH clients_without_identical_name AS (
    SELECT DISTINCT
        client.client_id,
        client.lda_name,
        client.normalized_name AS lda_normalized_name,
        client.lda_state
    FROM lda_clients AS client
    JOIN lda_client_match_status AS status USING (client_id)
    WHERE status.identical_name_eins = 0
),
blocked_candidates AS (
    SELECT
        client.client_id,
        client.lda_name,
        client.lda_normalized_name,
        client.lda_state,
        irs.ein,
        irs.irs_name,
        irs.normalized_name AS irs_normalized_name,
        irs.irs_state,
        jaro_winkler_similarity(
            client.lda_normalized_name,
            irs.normalized_name
        ) AS name_similarity
    FROM clients_without_identical_name AS client
    JOIN irs_entities AS irs
      ON LEFT(client.lda_normalized_name, 8) = LEFT(irs.normalized_name, 8)
     AND LEAST(
            LENGTH(client.lda_normalized_name),
            LENGTH(irs.normalized_name)
         )::DOUBLE
         / GREATEST(
            LENGTH(client.lda_normalized_name),
            LENGTH(irs.normalized_name)
         ) >= 0.60
    WHERE jaro_winkler_similarity(
        client.lda_normalized_name,
        irs.normalized_name
    ) >= {NEAR_NAME_MIN_SIMILARITY}
),
deduplicated AS (
    SELECT *,
        CASE
            WHEN irs_state = lda_state AND irs_state IS NOT NULL THEN 'State agrees'
            WHEN irs_state IS NULL OR lda_state IS NULL THEN 'State unavailable'
            ELSE 'State conflicts'
        END AS state_evidence,
        name_similarity
          + CASE
                WHEN irs_state = lda_state AND irs_state IS NOT NULL THEN 0.03
                WHEN irs_state IS NOT NULL AND lda_state IS NOT NULL THEN -0.03
                ELSE 0.00
            END AS ranking_score,
        ROW_NUMBER() OVER (
            PARTITION BY client_id, ein
            ORDER BY name_similarity DESC, lda_normalized_name
        ) AS ein_row
    FROM blocked_candidates
)
SELECT * EXCLUDE (ein_row)
FROM deduplicated
WHERE ein_row = 1
""")

conn.execute(f"""
CREATE OR REPLACE TEMP TABLE best_near_name_candidates AS
WITH ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY client_id
            ORDER BY ranking_score DESC, name_similarity DESC, ein
        ) AS candidate_rank,
        COUNT(*) OVER (PARTITION BY client_id) AS candidate_eins
    FROM near_name_candidates
)
SELECT *,
    CASE
        WHEN name_similarity >= {NEAR_NAME_STRONG_SIMILARITY}
         AND state_evidence = 'State agrees'
            THEN 'Strong near-name candidate: state agrees'
        WHEN name_similarity >= {NEAR_NAME_STRONG_SIMILARITY}
         AND state_evidence = 'State unavailable'
            THEN 'Strong near-name candidate: state unavailable'
        WHEN name_similarity >= {NEAR_NAME_CONFLICT_SIMILARITY}
         AND state_evidence = 'State conflicts'
            THEN 'Strong name candidate: state conflict requires review'
        ELSE 'Near-name candidate: manual review'
    END AS review_tier
FROM ranked
WHERE candidate_rank = 1
""")

near_match_summary = q("""
SELECT
    review_tier,
    COUNT(*) AS lda_clients,
    COUNT(*) FILTER (WHERE candidate_eins = 1) AS unique_candidate_clients,
    ROUND(AVG(name_similarity), 3) AS average_name_similarity
FROM best_near_name_candidates
GROUP BY review_tier
ORDER BY review_tier
""")

near_match_examples = q("""
SELECT
    client_id,
    lda_name,
    lda_state,
    ein,
    irs_name,
    irs_state,
    ROUND(name_similarity, 6) AS name_similarity,
    state_evidence,
    candidate_eins,
    review_tier
FROM best_near_name_candidates
ORDER BY name_similarity DESC, client_id
LIMIT 20
""")

display(near_match_summary.style.format({
    "lda_clients": "{:,.0f}",
    "unique_candidate_clients": "{:,.0f}",
    "average_name_similarity": "{:.3f}",
}))
display(near_match_examples)

,review_tier,lda_clients,unique_candidate_clients,average_name_similarity
0,Near-name candidate: manual review,"3,551","1,591",0.920
1,Strong name candidate: state conflict requires review,12,5,0.981
2,Strong near-name candidate: state agrees,100,42,0.956
3,Strong near-name candidate: state unavailable,"1,691",437,0.957


,client_id,lda_name,lda_state,ein,irs_name,irs_state,name_similarity,state_evidence,candidate_eins,review_tier
0,143445,INTERNATIONAL ASSOCIATION OF SHEET METAL AIR R...,DC,942693074,International Association of Sheet Metal Air R...,None,0.997101,State unavailable,134,Strong near-name candidate: state unavailable
1,211375,"INTERNATIONAL ASSOCIATION OF SHEET METAL, AIR,...",DC,942693074,International Association of Sheet Metal Air R...,None,0.997101,State unavailable,134,Strong near-name candidate: state unavailable
2,53424,"INTERNATIONAL ASSOCIATION OF SHEET METAL, AIR,...",DC,942693074,International Association of Sheet Metal Air R...,None,0.997101,State unavailable,134,Strong near-name candidate: state unavailable
3,58931,NATIONAL OPEN COMMERCE AND SAFER HIGHWAYS COAL...,OH,991388967,National Open Commerce and Safer Highway Coali...,None,0.995745,State unavailable,1,Strong near-name candidate: state unavailable
4,56648,"AMERICAN HEALTHY ALTERNATIVES ASSOCIATION, INC.",FL,882211047,AMERICAN HEALTHY ALTERNATIVES ASSOCIATIO,None,0.995122,State unavailable,7,Strong near-name candidate: state unavailable
5,60627,AMERICAN HEALTHY ALTERNATIVES ASSOCIATION,FL,882211047,AMERICAN HEALTHY ALTERNATIVES ASSOCIATIO,None,0.995122,State unavailable,7,Strong near-name candidate: state unavailable
6,132781,MID-WEST ELECTRIC CONSUMERS ASSOCIATION,CO,840509417,Mid-West Electric Consumers Associations Inc,None,0.995000,State unavailable,1,Strong near-name candidate: state unavailable
7,173941,AMERICAN EXPLORATION & MINING ASSOCIATION,DC,910491475,American Exploration & Mining Associatio,None,0.994872,State unavailable,9,Strong near-name candidate: state unavailable
8,188368,AMERICAN EXPLORATION & MINING ASSOCIATION,WA,910491475,American Exploration & Mining Associatio,None,0.994872,State unavailable,9,Strong near-name candidate: state unavailable
9,58973,UNDIAGNOSED DISEASE NETWORK FOUNDATION,DC,873474254,UNDIAGNOSED DISEASES NETWORK FOUNDATION,None,0.994872,State unavailable,1,Strong near-name candidate: state unavailable


In [ ]:
near_candidate_client_ids = set(
    q("SELECT DISTINCT client_id FROM best_near_name_candidates")["client_id"]
)

still_without_near_candidate = exact_name_misses.loc[
    ~exact_name_misses["client_id"].isin(near_candidate_client_ids)
].copy()
still_without_near_candidate["candidate_status"] = (
    "No near-name candidate (current rules)"
)

near_candidate_count = len(near_candidate_client_ids)
still_without_near_candidate_count = len(still_without_near_candidate)

print(f"Exact-name misses evaluated: {len(exact_name_misses):,}")
print(f"Clients with at least one near-name candidate: {near_candidate_count:,}")
print(
    "Clients still without a candidate under current rules: "
    f"{still_without_near_candidate_count:,}"
)
print(
    f"Displaying first 100 of {still_without_near_candidate_count:,} unresolved records "
    "(full dataset in still_without_near_candidate)."
)
display(still_without_near_candidate.head(100))

Exact-name misses evaluated: 23,030
Clients with at least one near-name candidate: 5,354
Clients still without a candidate under current rules: 17,676
Displaying first 100 of 17,676 unresolved records (full dataset in still_without_near_candidate).


,client_id,lda_name,lda_normalized_name,lda_state,lda_filings,first_filing_year,latest_filing_year,registrants,candidate_status
2,200360,BROWN-FORMAN CORPORATION,brown forman,KY,25,2023,2024,1,No near-name candidate (current rules)
3,131182,MASTERCARD WORLDWIDE,mastercard worldwide,DC,21,2023,2024,1,No near-name candidate (current rules)
4,199655,"TICHENOR VENTURES, LLC",tichenor ventures,TX,21,2023,2024,1,No near-name candidate (current rules)
7,207267,"PTC THERAPEUTICS, INC.",ptc therapeutics,NJ,18,2023,2024,1,No near-name candidate (current rules)
9,54868,"KODIAK AI, INC. (FORMERLY KODIAK ROBOTICS)",kodiak ai formerly kodiak robotics,CA,17,2023,2024,1,No near-name candidate (current rules)
...,...,...,...,...,...,...,...,...,...
141,153921,TERUMO BCT,terumo bct,CO,12,2023,2024,1,No near-name candidate (current rules)
142,205326,"THE CORMAC GROUP, LLC ON BEHALF OF BELL LEGAL ...",cormac group on behalf bell legal group,DC,12,2023,2024,1,No near-name candidate (current rules)
144,206316,THE TOWN OF PEMBROKE,town pembroke,NC,12,2023,2024,1,No near-name candidate (current rules)
147,197804,ULTRA MARITIME LLC (FORMERLY REPORTED AS ULTRA...,ultra maritime formerly reported as ultra elec...,CT,12,2023,2024,1,No near-name candidate (current rules)


In [ ]:
linkage_funnel = q("""
WITH counts AS (
    SELECT
        COUNT(*) AS total_clients,
        COUNT(*) FILTER (WHERE identical_name_eins > 0) AS identical_name_clients,
        (SELECT COUNT(*) FROM best_near_name_candidates) AS near_name_clients
    FROM lda_client_match_status
)
SELECT 1 AS sort_order, 'Total LDA clients' AS linkage_stage, total_clients AS lda_clients
FROM counts
UNION ALL
SELECT 2, 'Identical normalized-name candidate', identical_name_clients
FROM counts
UNION ALL
SELECT 3, 'Near-name candidate after exact-name miss', near_name_clients
FROM counts
UNION ALL
SELECT 4, 'No candidate under current rules',
       total_clients - identical_name_clients - near_name_clients
FROM counts
ORDER BY sort_order
""").drop(columns="sort_order")

total_clients = int(linkage_funnel.iloc[0]["lda_clients"])
linkage_funnel["share_of_all_clients"] = (
    linkage_funnel["lda_clients"] / total_clients
 )

classified_clients = int(linkage_funnel.iloc[1:]["lda_clients"].sum())
assert classified_clients == total_clients

display(linkage_funnel.style.format({
    "lda_clients": "{:,.0f}",
    "share_of_all_clients": "{:.1%}",
}))
print(
    "The final three rows are mutually exclusive outcomes and sum to all LDA clients."
)

,linkage_stage,lda_clients,share_of_all_clients
0,Total LDA clients,"28,790",100.0%
1,Identical normalized-name candidate,"5,760",20.0%
2,Near-name candidate after exact-name miss,"5,354",18.6%
3,No candidate under current rules,"17,676",61.4%


The final three rows are mutually exclusive outcomes and sum to all LDA clients.


## Interpretation

Use **Tier A** as the narrow primary entity-link cohort: the normalized name maps to one IRS EIN and IRS/LDA states agree. Use **Tier A+B** as a sensitivity cohort: Tier B allows state to resolve a normalized name that initially maps to multiple EINs. Tier C, D, and E remain organization-level candidates but should not be included automatically in EIN-specific financial analysis.

Near-name results are review candidates, not approved EIN links. Review them using state, alternate or historical names, and addresses when available. State agreement strengthens a candidate but does not replace name evidence. Likewise, no candidate under these rules does not prove that an organization is absent from the Form 990 population; the LDA universe also includes for-profit companies and public entities outside Form 990's scope.

The current near-name block requires the same first eight normalized characters, so it will miss acronyms and names whose leading words changed. Avoid globally removing identity-bearing words such as `association`; use a separate alias or token-based candidate stage for those cases.